# 0. Setup & Configuration

## Objective
Initialize the Spark functions and define the source and target locations
used throughout the Sales Data ETL pipeline.

## Why this stage matters
Keeping imports and configuration in one place makes the notebook
easier to understand, maintain, and reuse.

In [0]:
# ==========================================
# 0. Setup & Configuration
# ==========================================

# Import PySpark SQL functions for transformations,
# filtering, aggregations, and data-quality checks.
import pyspark.sql.functions as F

In [0]:
# ==========================================
# 0. Setup & Configuration
# ==========================================

# Define the catalog and schema used for this project.
catalog_name = "de_learning"
schema_name = "sales_etl"

# Define the source directory containing the raw CSV files.
source_path = f"/Volumes/{catalog_name}/{schema_name}/raw_files"

# Define individual source file paths.
customers_path = f"{source_path}/customers.csv"
orders_path = f"{source_path}/orders.csv"
products_path = f"{source_path}/products.csv"

In [0]:
print(customers_path)
print(orders_path)
print(products_path)

/Volumes/de_learning/sales_etl/raw_files/customers.csv
/Volumes/de_learning/sales_etl/raw_files/orders.csv
/Volumes/de_learning/sales_etl/raw_files/products.csv


# 1. Customer Data Exploration

## Objective
Load the customer source data and understand its structure and quality before using it in downstream transformations.

## What we'll check
- Load the customer CSV file into a Spark DataFrame.
- Inspect the schema, columns, and record count.
- Identify missing values, duplicate customer IDs, and unwanted whitespace.
- Create a cleaned customer DataFrame for later joins.

## Expected outcome
Understand the quality of the customer master data and prepare a cleaned version for integration with sales orders.

In [0]:
# ==========================================
# 1. Customer Data Exploration
# ==========================================

# Load the customer CSV file into a Spark DataFrame.
# header=True uses the first row as column names.
# inferSchema=True asks Spark to infer column data types.
customers_df = spark.read.csv(
    customers_path,
    header=True,
    inferSchema=True
)

In [0]:
# ------------------------------------------
# 1.1 Basic Data Inspection
# ------------------------------------------

# Display the first 10 records to understand the data.
customers_df.show(10, truncate=False)

# Print the schema to inspect column names and data types.
customers_df.printSchema()

# Count the total number of customer records loaded.
print("Total customer records:", customers_df.count())

# Display the column names available in the DataFrame.
print("Column names:", customers_df.columns)

+-----------+--------------+---------+-------+
|customer_id|customer_name |city     |country|
+-----------+--------------+---------+-------+
|CUST0001   |Sai Sharma    |Kochi    |India  |
|CUST0002   |Riya Khan     |Hyderabad|India  |
|CUST0003   |Ayaan Anderson|Delhi    |India  |
|CUST0004   |Noah Patel    |Lucknow  |India  |
|CUST0005   |Priya Verma   |Mumbai   |India  |
|CUST0006   |Reyansh Iyer  |Hyderabad|India  |
|CUST0007   |Liam Jones    |Mumbai   |India  |
|CUST0008   |Sophia Iyer   |NULL     |India  |
|CUST0009   |Noah Bhat     |Hyderabad|India  |
|CUST0010   |Karan Brown   |Chennai  |India  |
+-----------+--------------+---------+-------+
only showing top 10 rows
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)

Total customer records: 100
Column names: ['customer_id', 'customer_name', 'city', 'country']


In [0]:
# ------------------------------------------
# 1.2 Customer Data Quality Checks
# ------------------------------------------

# Check for missing values in important customer fields.
customers_df.filter(
    F.col("city").isNull() |
    F.col("country").isNull()
).show(truncate=False)

# Check whether any customer ID appears more than once.
customers_df.groupBy("customer_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

# Identify text values with leading or trailing whitespace.
customers_df.filter(
    (F.col("customer_name") != F.trim(F.col("customer_name"))) |
    (F.col("city") != F.trim(F.col("city"))) |
    (F.col("country") != F.trim(F.col("country")))
).show(truncate=False)

+-----------+--------------+-------------+-------+
|customer_id|customer_name |city         |country|
+-----------+--------------+-------------+-------+
|CUST0008   |Sophia Iyer   |NULL         |India  |
|CUST0019   |Noah Mehta    |Visakhapatnam|NULL   |
|CUST0042   |Rohan Iyer    |NULL         |India  |
|CUST0056   |Sophia Jackson|Mumbai       |NULL   |
|CUST0073   |Isha Khan     |NULL         |India  |
|CUST0091   |Vihaan Khan   |Delhi        |NULL   |
|CUST0099   |Muhammad Davis|NULL         |NULL   |
+-----------+--------------+-------------+-------+

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+

+-----------+---------------+----------+--------------------+
|customer_id|customer_name  |city      |country             |
+-----------+---------------+----------+--------------------+
|CUST0012   |  Diya Wilson  |Pune      |India               |
|CUST0015   |Muhammad Patel | Pune     |India               |
|CUST0023   |Sneha Brown    |Hyderabad |India  

In [0]:
# ------------------------------------------
# 1.3 Clean Customer Text Columns
# ------------------------------------------

# Create a cleaned DataFrame by trimming leading
# and trailing whitespace from text columns.
customers_clean_df = customers_df

for c in ["customer_name", "city", "country"]:
    customers_clean_df = customers_clean_df.withColumn(
        c,
        F.trim(F.col(c))
    )

In [0]:
# ------------------------------------------
# 1.4 Validate Cleaned Customer Data
# ------------------------------------------

# Check whether any leading or trailing whitespace remains.
customers_clean_df.filter(
    (F.col("customer_name") != F.trim(F.col("customer_name"))) |
    (F.col("city") != F.trim(F.col("city"))) |
    (F.col("country") != F.trim(F.col("country")))
).show(truncate=False)

# Confirm the record count remains unchanged.
print("Original customer count:", customers_df.count())
print("Cleaned customer count:", customers_clean_df.count())

# Verify that missing city and country values remain.
customers_clean_df.filter(
    F.col("city").isNull() |
    F.col("country").isNull()
).show(truncate=False)

+-----------+-------------+----+-------+
|customer_id|customer_name|city|country|
+-----------+-------------+----+-------+
+-----------+-------------+----+-------+

Original customer count: 100
Cleaned customer count: 100
+-----------+--------------+-------------+-------+
|customer_id|customer_name |city         |country|
+-----------+--------------+-------------+-------+
|CUST0008   |Sophia Iyer   |NULL         |India  |
|CUST0019   |Noah Mehta    |Visakhapatnam|NULL   |
|CUST0042   |Rohan Iyer    |NULL         |India  |
|CUST0056   |Sophia Jackson|Mumbai       |NULL   |
|CUST0073   |Isha Khan     |NULL         |India  |
|CUST0091   |Vihaan Khan   |Delhi        |NULL   |
|CUST0099   |Muhammad Davis|NULL         |NULL   |
+-----------+--------------+-------------+-------+



In [0]:


customers_clean_df.agg(
    F.count("*").alias("total_records"),
    F.countDistinct("customer_id").alias("distinct_customer_ids"),
    F.sum(F.when(F.col("city").isNull(), 1).otherwise(0)).alias("missing_city"),
    F.sum(F.when(F.col("country").isNull(), 1).otherwise(0)).alias("missing_country"),
    F.sum(
        F.when(
            F.col("city").isNull() | F.col("country").isNull(),
            1
        ).otherwise(0)
    ).alias("records_with_missing_location")
).show()


+-------------+---------------------+------------+---------------+-----------------------------+
|total_records|distinct_customer_ids|missing_city|missing_country|records_with_missing_location|
+-------------+---------------------+------------+---------------+-----------------------------+
|          100|                  100|           4|              4|                            7|
+-------------+---------------------+------------+---------------+-----------------------------+



# 2. Order Data Exploration

## Objective
Load the order source data and assess its structure, completeness, uniqueness, and validity before integrating it with customer and product data.

## What we'll check
- Load the orders CSV file into a Spark DataFrame.
- Inspect the schema, columns, and record count.
- Identify missing values and duplicate order records.
- Validate order quantities and key identifiers.
- Prepare a cleaned dataset for downstream transformations.

## Expected outcome
Understand the quality issues in the order data and establish the rules for creating a valid set of orders.


In [0]:
# ==========================================
# 2. Order Data Exploration
# ==========================================

# Load the orders CSV file into a Spark DataFrame.
# header=True reads column names from the first row.
# inferSchema=True asks Spark to infer column data types.
orders_df = spark.read.csv(
    orders_path,
    header=True,
    inferSchema=True
)

In [0]:
# ------------------------------------------
# 2.1 Basic Data Inspection
# ------------------------------------------

# Display the first 10 orders.
orders_df.show(10, truncate=False)

# Inspect column names and data types.
orders_df.printSchema()

# Count the total number of order records.
print("Total order records:", orders_df.count())

# Display the available column names.
print("Column names:", orders_df.columns)

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+----------+--------+----------+
|ORD00001|CUST0060   |PRD028    |1       |2025-04-16|
|ORD00002|CUST0084   |PRD021    |1       |2025-06-22|
|ORD00003|CUST0052   |PRD024    |3       |2025-01-16|
|ORD00004|CUST0032   |PRD007    |2       |2025-01-28|
|ORD00005|CUST0058   |PRD005    |4       |2025-05-18|
|ORD00006|CUST0036   |PRD015    |2       |2025-02-16|
|ORD00007|CUST0057   |PRD026    |5       |2025-01-20|
|ORD00008|CUST0007   |PRD021    |5       |2025-01-26|
|ORD00009|CUST0012   |PRD030    |2       |2025-01-04|
|ORD00010|CUST0053   |PRD016    |4       |2025-02-12|
+--------+-----------+----------+--------+----------+
only showing top 10 rows
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_date: date (nullable = true)

Total o

In [0]:
# ------------------------------------------
# 2.2 Missing Value Analysis
# ------------------------------------------

# Count missing values in each column.
orders_df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in orders_df.columns
]).show()

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+----------+--------+----------+
|       0|          3|         2|       0|         0|
+--------+-----------+----------+--------+----------+



In [0]:
# ------------------------------------------
# 2.3 Duplicate Order ID Check
# ------------------------------------------

# Group records by order_id and count occurrences.
# Show only IDs that appear more than once.
orders_df.groupBy("order_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(truncate=False)

+--------+-----+
|order_id|count|
+--------+-----+
|ORD00451|2    |
|ORD00016|2    |
|ORD00151|2    |
|ORD00221|2    |
|ORD00081|2    |
|ORD00306|2    |
|ORD00381|2    |
+--------+-----+



In [0]:
# ------------------------------------------
# 2.4 Inspect Duplicate Order Records
# ------------------------------------------

# Find order IDs that occur more than once.
duplicate_order_ids_df = (
    orders_df.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .select("order_id")
)

# Retrieve every source row associated with those IDs.
duplicate_orders_df = orders_df.join(
    duplicate_order_ids_df,
    on="order_id",
    how="inner"
)

# Display duplicate rows in order.
duplicate_orders_df.orderBy("order_id").show(truncate=False)

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+----------+--------+----------+
|ORD00016|CUST0038   |PRD007    |1       |2025-02-18|
|ORD00016|CUST0038   |PRD007    |1       |2025-02-18|
|ORD00081|CUST0056   |PRD026    |5       |2025-02-23|
|ORD00081|CUST0056   |PRD026    |5       |2025-02-23|
|ORD00151|CUST0097   |PRD015    |5       |2025-03-04|
|ORD00151|CUST0097   |PRD015    |5       |2025-03-04|
|ORD00221|CUST0057   |PRD029    |4       |2025-03-15|
|ORD00221|CUST0057   |PRD029    |4       |2025-03-15|
|ORD00306|CUST0088   |PRD012    |5       |2025-01-26|
|ORD00306|CUST0088   |PRD012    |5       |2025-01-26|
|ORD00381|CUST0091   |PRD021    |4       |2025-04-14|
|ORD00381|CUST0091   |PRD021    |4       |2025-04-14|
|ORD00451|CUST0090   |PRD027    |2       |2025-05-01|
|ORD00451|CUST0090   |PRD027    |2       |2025-05-01|
+--------+-----------+----------+--------+----------+



In [0]:
# ------------------------------------------
# 2.5 Remove Exact Duplicate Records
# ------------------------------------------

# Remove rows that are identical across all columns.
orders_dedup_df = orders_df.dropDuplicates()

# Compare record counts before and after deduplication.
print("Original order records:", orders_df.count())
print("Records after deduplication:", orders_dedup_df.count())
print(
    "Duplicate records removed:",
    orders_df.count() - orders_dedup_df.count()
)

Original order records: 500
Records after deduplication: 493
Duplicate records removed: 7


In [0]:
# ------------------------------------------
# 2.6 Invalid Quantity Check
# ------------------------------------------

# Find orders with zero or negative quantities.
invalid_quantity_df = orders_dedup_df.filter(
    F.col("quantity") <= 0
)

# Display the invalid records for investigation.
invalid_quantity_df.show(truncate=False)

# Count the records that violate the quantity rule.
print(
    "Invalid quantity records:",
    invalid_quantity_df.count()
)

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+----------+--------+----------+
|ORD00290|CUST0048   |PRD022    |0       |2025-04-27|
|ORD00440|CUST0033   |PRD002    |-2      |2025-02-05|
|ORD00050|CUST0006   |PRD010    |0       |2025-02-02|
|ORD00180|CUST0074   |PRD021    |-1      |2025-01-19|
+--------+-----------+----------+--------+----------+

Invalid quantity records: 4


In [0]:
# ------------------------------------------
# 2.7 Missing Identifier Investigation
# ------------------------------------------

# Find orders that are missing a customer or product ID.
missing_identifier_df = orders_dedup_df.filter(
    F.col("customer_id").isNull() |
    F.col("product_id").isNull()
)

# Display affected orders for investigation.
missing_identifier_df.show(truncate=False)

# Count affected records.
print(
    "Orders with missing identifiers:",
    missing_identifier_df.count()
)

+--------+-----------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|order_date|
+--------+-----------+----------+--------+----------+
|ORD00025|NULL       |PRD019    |5       |2025-06-18|
|ORD00068|CUST0056   |NULL      |5       |2025-03-22|
|ORD00140|NULL       |PRD026    |1       |2025-06-25|
|ORD00312|NULL       |PRD027    |5       |2025-02-22|
|ORD00275|CUST0078   |NULL      |3       |2025-02-28|
+--------+-----------+----------+--------+----------+

Orders with missing identifiers: 5


In [0]:
# ------------------------------------------
# 2.8 Create Valid Orders Dataset
# ------------------------------------------

# Keep only orders with both identifiers present
# and a positive quantity.
orders_final_valid_df = orders_dedup_df.filter(
    F.col("customer_id").isNotNull() &
    F.col("product_id").isNotNull() &
    (F.col("quantity") > 0)
)

# Compare record counts before and after validation.
print("Original order records:", orders_df.count())
print("After deduplication:", orders_dedup_df.count())
print("Valid order records:", orders_final_valid_df.count())
print(
    "Records excluded during validation:",
    orders_dedup_df.count() - orders_final_valid_df.count()
)

Original order records: 500
After deduplication: 493
Valid order records: 484
Records excluded during validation: 9


# 3. Product Data Exploration

## Objective
Load the product master data and assess its structure, completeness, uniqueness, and price validity before joining it to orders.

## What we'll check
- Load the products CSV file into a Spark DataFrame.
- Inspect the schema, columns, and record count.
- Check for missing values and duplicate product IDs.
- Validate unit prices.

## Expected outcome
Understand the quality of the product master data and prepare it for integration with valid orders.


In [0]:
# ==========================================
# 3. Product Data Exploration
# ==========================================

# Load the product CSV file into a Spark DataFrame.
products_df = spark.read.csv(
    products_path,
    header=True,
    inferSchema=True
)

In [0]:
# ------------------------------------------
# 3.1 Basic Data Inspection
# ------------------------------------------

# Display the first 10 products.
products_df.show(10, truncate=False)

# Inspect column names and data types.
products_df.printSchema()

# Count the total number of product records.
print("Total product records:", products_df.count())

# Display the available column names.
print("Column names:", products_df.columns)

+----------+-----------------------------------+-----------+----------+
|product_id|product_name                       |category   |unit_price|
+----------+-----------------------------------+-----------+----------+
|PRD001    |Wireless Noise-Canceling Headphones|Electronics|7999.0    |
|PRD002    |Mechanical Gaming Keyboard         |Electronics|4499.5    |
|PRD003    |Ergonomic Optical Mouse            |Electronics|1299.0    |
|PRD004    |27-inch 4K UHD Monitor             |Electronics|24999.99  |
|PRD005    |Smartwatch Series 5                |Electronics|15499.0   |
|PRD006    |USB-C Multiport Adapter Hub        |Electronics|2199.0    |
|PRD007    |Portable Bluetooth Speaker         |Electronics|3499.0    |
|PRD008    |Ergonomic Mesh Office Chair        |Furniture  |11500.0   |
|PRD009    |Solid Oak Study Desk               |Furniture  |16800.0   |
|PRD010    |Adjustable Standing Desk Riser     |Furniture  |8990.0    |
+----------+-----------------------------------+-----------+----

In [0]:
# ------------------------------------------
# 3.2 Missing Value Analysis
# ------------------------------------------

# Count missing values in each product column.
products_df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in products_df.columns
]).show()

+----------+------------+--------+----------+
|product_id|product_name|category|unit_price|
+----------+------------+--------+----------+
|         0|           0|       0|         0|
+----------+------------+--------+----------+



In [0]:
# ------------------------------------------
# 3.3 Duplicate Product ID Check
# ------------------------------------------

# Group products by product_id and count occurrences.
# Show only IDs that appear more than once.
products_df.groupBy("product_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(truncate=False)

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



In [0]:
# ------------------------------------------
# 3.4 Unit Price Validation
# ------------------------------------------

# Find products with zero or negative unit prices.
invalid_price_df = products_df.filter(
    F.col("unit_price") <= 0
)

# Display any products with invalid prices.
invalid_price_df.show(truncate=False)

# Count invalid price records.
print("Invalid price records:", invalid_price_df.count())

+----------+------------+--------+----------+
|product_id|product_name|category|unit_price|
+----------+------------+--------+----------+
+----------+------------+--------+----------+

Invalid price records: 0


# 4. Orders & Products Integration

## Objective
Enrich valid order records with product details and calculate the sales amount for each order.

## What we'll do
- Join valid orders with product data using product_id.
- Identify orders that do not match a product record.
- Calculate sales_amount using quantity and unit_price.
- Validate row counts and investigate unmatched records.

## Expected outcome
Create an enriched order dataset with product information and identify orders that require product-data investigation.


In [0]:
# ==========================================
# 4. Orders & Products Integration
# ==========================================

# Join valid orders with product details.
# A left join preserves every valid order, even if
# its product_id has no matching product record.
orders_with_products_df = orders_final_valid_df.join(
    products_df,
    on="product_id",
    how="left"
)

# Validate the row count after joining.
print("Valid orders:", orders_final_valid_df.count())
print("Orders after product join:", orders_with_products_df.count())

Valid orders: 484
Orders after product join: 484


In [0]:
# ------------------------------------------
# 4.1 Identify Unmatched Products
# ------------------------------------------

# Find orders where no matching product name was found.
unmatched_products_df = orders_with_products_df.filter(
    F.col("product_name").isNull()
)

# Display the unmatched orders for investigation.
unmatched_products_df.show(truncate=False)

# Count unmatched orders.
print("Unmatched product orders:", unmatched_products_df.count())

# Count orders with matching product records.
print(
    "Matched product orders:",
    orders_with_products_df.filter(
        F.col("product_name").isNotNull()
    ).count()
)

+----------+--------+-----------+--------+----------+------------+--------+----------+
|product_id|order_id|customer_id|quantity|order_date|product_name|category|unit_price|
+----------+--------+-----------+--------+----------+------------+--------+----------+
|PRD920    |ORD00360|CUST0054   |3       |2025-04-07|NULL        |NULL    |NULL      |
|PRD934    |ORD00115|CUST0029   |2       |2025-06-04|NULL        |NULL    |NULL      |
+----------+--------+-----------+--------+----------+------------+--------+----------+

Unmatched product orders: 2
Matched product orders: 482


In [0]:
# ------------------------------------------
# 4.2 Calculate Sales Amount
# ------------------------------------------

# Calculate sales amount for each order.
# Orders without a matched unit price will have a null amount.
orders_with_products_df = orders_with_products_df.withColumn(
    "sales_amount",
    F.col("quantity") * F.col("unit_price")
)

# Display sample records with their calculated sales amounts.
orders_with_products_df.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "sales_amount"
).show(10, truncate=False)

# Count orders with missing sales amounts.
print(
    "Orders with missing sales amount:",
    orders_with_products_df.filter(
        F.col("sales_amount").isNull()
    ).count()
)

+--------+----------+--------+----------+------------------+
|order_id|product_id|quantity|unit_price|sales_amount      |
+--------+----------+--------+----------+------------------+
|ORD00022|PRD013    |1       |6300.0    |6300.0            |
|ORD00034|PRD027    |5       |1850.0    |9250.0            |
|ORD00098|PRD029    |4       |999.0     |3996.0            |
|ORD00100|PRD015    |2       |2899.5    |5799.0            |
|ORD00112|PRD016    |1       |3299.0    |3299.0            |
|ORD00127|PRD012    |2       |5499.0    |10998.0           |
|ORD00156|PRD018    |1       |1199.0    |1199.0            |
|ORD00157|PRD003    |2       |1299.0    |2598.0            |
|ORD00163|PRD017    |3       |899.0     |2697.0            |
|ORD00167|PRD004    |5       |24999.99  |124999.95000000001|
+--------+----------+--------+----------+------------------+
only showing top 10 rows
Orders with missing sales amount: 2


In [0]:
# ------------------------------------------
# 4.3 Validate Sales Amounts
# ------------------------------------------

# Count records with missing product details.
print(
    "Orders without product match:",
    orders_with_products_df.filter(
        F.col("product_name").isNull()
    ).count()
)

# Count records with missing sales amounts.
print(
    "Orders with missing sales amount:",
    orders_with_products_df.filter(
        F.col("sales_amount").isNull()
    ).count()
)

# Calculate total sales amount for matched products.
total_sales = orders_with_products_df.agg(
    F.sum("sales_amount").alias("total_sales")
).first()["total_sales"]

print("Total sales amount:", total_sales)

Orders without product match: 2
Orders with missing sales amount: 2
Total sales amount: 9539579.400000002


# 5. Customer, Orders & Products Integration

## Objective
Enrich the order dataset with customer details and identify orders that cannot be matched to the customer master.

## What we'll do
- Join orders and product details with the cleaned customer DataFrame.
- Preserve all valid orders using a left join.
- Identify orders with missing customer matches.
- Add a status column to distinguish valid customer matches from exceptions.
- Validate record counts and customer match results.

## Expected outcome
Create a combined sales dataset containing order, product, and customer details, with customer-data exceptions clearly identified.


In [0]:
# ==========================================
# 5. Customer, Orders & Products Integration
# ==========================================

# Join the enriched orders with cleaned customer details.
# A left join preserves every order in the input DataFrame.
sales_with_customers_df = orders_with_products_df.join(
    customers_clean_df,
    on="customer_id",
    how="left"
)

# Validate that the join preserves the order count.
print("Orders before customer join:", orders_with_products_df.count())
print("Orders after customer join:", sales_with_customers_df.count())

Orders before customer join: 484
Orders after customer join: 484


In [0]:
# ------------------------------------------
# 5.1 Identify Missing Customer Matches
# ------------------------------------------

# Find orders where no customer name was matched.
missing_customers_df = sales_with_customers_df.filter(
    F.col("customer_name").isNull()
)

# Display affected orders for investigation.
missing_customers_df.select(
    "order_id",
    "customer_id",
    "product_id",
    "product_name",
    "sales_amount"
).show(truncate=False)

# Count orders without a matching customer.
print(
    "Orders without customer match:",
    missing_customers_df.count()
)

# Count orders with a matching customer.
print(
    "Orders with customer match:",
    sales_with_customers_df.filter(
        F.col("customer_name").isNotNull()
    ).count()
)

+--------+-----------+----------+------------------------------+------------------+
|order_id|customer_id|product_id|product_name                  |sales_amount      |
+--------+-----------+----------+------------------------------+------------------+
|ORD00415|CUST9694   |PRD004    |27-inch 4K UHD Monitor        |124999.95000000001|
|ORD00210|CUST9356   |PRD003    |Ergonomic Optical Mouse       |5196.0            |
|ORD00085|CUST9679   |PRD010    |Adjustable Standing Desk Riser|44950.0           |
+--------+-----------+----------+------------------------------+------------------+

Orders without customer match: 3
Orders with customer match: 481


In [0]:
# ------------------------------------------
# 5.2 Add Customer Data Status
# ------------------------------------------

# Flag orders based on whether a customer match was found.
sales_with_customers_df = sales_with_customers_df.withColumn(
    "customer_data_status",
    F.when(
        F.col("customer_name").isNull(),
        "MISSING_CUSTOMER"
    ).otherwise("VALID")
)

# Summarize orders by customer data status.
sales_with_customers_df.groupBy(
    "customer_data_status"
).count().show()

+--------------------+-----+
|customer_data_status|count|
+--------------------+-----+
|               VALID|  481|
|    MISSING_CUSTOMER|    3|
+--------------------+-----+



# 6. Sales Data Quality & Validation

## Objective
Validate the integrated sales dataset and identify records
that cannot be included in the main sales dataset.

## What we'll do
- Identify orders with missing product details.
- Validate sales amount completeness.
- Review customer data quality status.
- Summarize the issues before separating main sales
  records from exception records.

## Expected outcome
Understand the quality of the integrated dataset before
creating the main sales dataset and exception datasets.


In [0]:

# ------------------------------------------
# 6.1 Validate the Integrated Dataset
# ------------------------------------------

# Check the total number of records after integration.
print(
    "Total integrated orders:",
    sales_with_customers_df.count()
)

# Count orders that have no matching product.
print(
    "Orders without product match:",
    sales_with_customers_df.filter(
        F.col("product_name").isNull()
    ).count()
)

# Count orders with missing sales amounts.
print(
    "Orders with missing sales amount:",
    sales_with_customers_df.filter(
        F.col("sales_amount").isNull()
    ).count()
)

# Count orders without a matching customer.
print(
    "Orders without customer match:",
    sales_with_customers_df.filter(
        F.col("customer_data_status") == "MISSING_CUSTOMER"
    ).count()
)


Total integrated orders: 484
Orders without product match: 2
Orders with missing sales amount: 2
Orders without customer match: 3


In [0]:

# ------------------------------------------
# 6.2 Review Customer Data Quality
# ------------------------------------------

# Summarize records by customer data status.
sales_with_customers_df.groupBy(
    "customer_data_status"
).count().show()

# Display orders that have no matching customer.
sales_with_customers_df.filter(
    F.col("customer_data_status") == "MISSING_CUSTOMER"
).select(
    "order_id",
    "customer_id",
    "product_id",
    "product_name",
    "sales_amount",
    "customer_data_status"
).show(truncate=False)


+--------------------+-----+
|customer_data_status|count|
+--------------------+-----+
|               VALID|  481|
|    MISSING_CUSTOMER|    3|
+--------------------+-----+

+--------+-----------+----------+------------------------------+------------------+--------------------+
|order_id|customer_id|product_id|product_name                  |sales_amount      |customer_data_status|
+--------+-----------+----------+------------------------------+------------------+--------------------+
|ORD00415|CUST9694   |PRD004    |27-inch 4K UHD Monitor        |124999.95000000001|MISSING_CUSTOMER    |
|ORD00210|CUST9356   |PRD003    |Ergonomic Optical Mouse       |5196.0            |MISSING_CUSTOMER    |
|ORD00085|CUST9679   |PRD010    |Adjustable Standing Desk Riser|44950.0           |MISSING_CUSTOMER    |
+--------+-----------+----------+------------------------------+------------------+--------------------+



In [0]:

# ------------------------------------------
# 6.3 Separate Main Sales and Product Exceptions
# ------------------------------------------

# Keep orders with matched product details in the main dataset.
main_sales_df = sales_with_customers_df.filter(
    F.col("product_name").isNotNull()
)

# Keep orders without matched product details for investigation.
product_exceptions_df = sales_with_customers_df.filter(
    F.col("product_name").isNull()
)

# Validate the number of records in each dataset.
print("Main sales records:", main_sales_df.count())
print("Product exception records:", product_exceptions_df.count())

# Confirm that the two datasets reconcile to the original count.
print(
    "Total records after separation:",
    main_sales_df.count() + product_exceptions_df.count()
)


Main sales records: 482
Product exception records: 2
Total records after separation: 484


In [0]:

# ------------------------------------------
# 6.4 Create Customer Exceptions
# ------------------------------------------

# Identify main sales records without a matching customer.
customer_exceptions_df = main_sales_df.filter(
    F.col("customer_data_status") == "MISSING_CUSTOMER"
)

# Display the customer exceptions for investigation.
customer_exceptions_df.select(
    "order_id",
    "customer_id",
    "product_id",
    "product_name",
    "sales_amount",
    "customer_data_status"
).show(truncate=False)

# Count the customer exception records.
print(
    "Customer exception records:",
    customer_exceptions_df.count()
)


+--------+-----------+----------+------------------------------+------------------+--------------------+
|order_id|customer_id|product_id|product_name                  |sales_amount      |customer_data_status|
+--------+-----------+----------+------------------------------+------------------+--------------------+
|ORD00415|CUST9694   |PRD004    |27-inch 4K UHD Monitor        |124999.95000000001|MISSING_CUSTOMER    |
|ORD00210|CUST9356   |PRD003    |Ergonomic Optical Mouse       |5196.0            |MISSING_CUSTOMER    |
|ORD00085|CUST9679   |PRD010    |Adjustable Standing Desk Riser|44950.0           |MISSING_CUSTOMER    |
+--------+-----------+----------+------------------------------+------------------+--------------------+

Customer exception records: 3


# 7. Main Sales Dataset & Exception Handling

## Objective
Prepare the final main sales dataset and organize exception records for investigation.

## What we'll do
- Review the columns and structure of the main sales dataset.
- Validate the main sales records.
- Review product and customer exception datasets.
- Reconcile the datasets to confirm record counts.

## Expected outcome
Prepare a validated main sales dataset and clearly separated exception datasets for storage and downstream reporting.


In [0]:

# ------------------------------------------
# 7.1 Inspect the Main Sales Dataset
# ------------------------------------------

# Display sample records from the main sales dataset.
main_sales_df.show(10, truncate=False)

# Inspect the column names and data types.
main_sales_df.printSchema()

# Confirm the total number of main sales records.
print("Main sales records:", main_sales_df.count())

# Display the available column names.
print("Main sales columns:", main_sales_df.columns)


+-----------+----------+--------+--------+----------+-------------------------------+-----------+----------+------------------+-------------+-------------+--------------------+--------------------+
|customer_id|product_id|order_id|quantity|order_date|product_name                   |category   |unit_price|sales_amount      |customer_name|city         |country             |customer_data_status|
+-----------+----------+--------+--------+----------+-------------------------------+-----------+----------+------------------+-------------+-------------+--------------------+--------------------+
|CUST0031   |PRD013    |ORD00022|1       |2025-06-22|Memory Foam Floor Recliner     |Furniture  |6300.0    |6300.0            |Riya Wilson  |Visakhapatnam|India               |VALID               |
|CUST0057   |PRD027    |ORD00034|5       |2025-02-10|Polarized Aviator Sunglasses   |Accessories|1850.0    |9250.0            |Sai Davis    |Visakhapatnam|India               |VALID               |
|CUST0097 

In [0]:

# ------------------------------------------
# 7.2 Validate Main Sales Records
# ------------------------------------------

# Check whether any main sales record has missing product details.
print(
    "Missing product names:",
    main_sales_df.filter(
        F.col("product_name").isNull()
    ).count()
)

# Check whether any main sales record has a missing sales amount.
print(
    "Missing sales amounts:",
    main_sales_df.filter(
        F.col("sales_amount").isNull()
    ).count()
)

# Check for non-positive quantities.
print(
    "Invalid quantities:",
    main_sales_df.filter(
        F.col("quantity") <= 0
    ).count()
)

# Check whether every record has a customer data status.
print(
    "Missing customer status:",
    main_sales_df.filter(
        F.col("customer_data_status").isNull()
    ).count()
)


Missing product names: 0
Missing sales amounts: 0
Invalid quantities: 0
Missing customer status: 0


In [0]:

# ------------------------------------------
# 7.3 Validate Exception Datasets
# ------------------------------------------

# Confirm that product exceptions have missing product details.
print(
    "Product exceptions with missing product name:",
    product_exceptions_df.filter(
        F.col("product_name").isNull()
    ).count()
)

# Confirm that product exceptions have missing sales amounts.
print(
    "Product exceptions with missing sales amount:",
    product_exceptions_df.filter(
        F.col("sales_amount").isNull()
    ).count()
)

# Confirm that customer exceptions have the expected status.
print(
    "Customer exceptions with missing-customer status:",
    customer_exceptions_df.filter(
        F.col("customer_data_status") == "MISSING_CUSTOMER"
    ).count()
)

# Display the number of records in each exception dataset.
print("Total product exceptions:", product_exceptions_df.count())
print("Total customer exceptions:", customer_exceptions_df.count())


Product exceptions with missing product name: 2
Product exceptions with missing sales amount: 2
Customer exceptions with missing-customer status: 3
Total product exceptions: 2
Total customer exceptions: 3


In [0]:

# ------------------------------------------
# 7.4 Reconcile Dataset Record Counts
# ------------------------------------------

# Count records in the main sales and product exception datasets.
main_sales_count = main_sales_df.count()
product_exceptions_count = product_exceptions_df.count()

# Calculate the total number of records accounted for.
reconciled_count = main_sales_count + product_exceptions_count

# Compare with the original integrated dataset.
integrated_count = sales_with_customers_df.count()

print("Integrated records:", integrated_count)
print("Main sales records:", main_sales_count)
print("Product exception records:", product_exceptions_count)
print("Reconciled records:", reconciled_count)
print("Counts match:", integrated_count == reconciled_count)


Integrated records: 484
Main sales records: 482
Product exception records: 2
Reconciled records: 484
Counts match: True


# 8. Delta Table Creation & Validation

## Objective
Persist the validated sales datasets as Delta tables and verify that the stored data matches the source DataFrames.

## What we'll do
- Save the main sales dataset as a Delta table.
- Save product exceptions and customer exceptions as separate Delta tables.
- Validate the stored tables by checking their record counts.
- Confirm that the stored main sales and product exception tables reconcile to the integrated dataset.

## Expected outcome
Create reusable Delta tables for the main sales dataset and both exception datasets, ready for downstream queries and reporting.


In [0]:

# ------------------------------------------
# 8.1 Save the Main Sales Delta Table
# ------------------------------------------

# Define the destination table for the main sales dataset.
main_sales_table = f"{catalog_name}.{schema_name}.main_sales"

# Save the validated DataFrame as a managed Delta table.
# Overwrite mode refreshes the table when this cell is rerun.
main_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(main_sales_table)

print("Main sales table saved:", main_sales_table)


Main sales table saved: de_learning.sales_etl.main_sales


In [0]:

# ------------------------------------------
# 8.2 Validate the Main Sales Delta Table
# ------------------------------------------

# Read the saved Delta table back from the catalog.
main_sales_table_df = spark.table(main_sales_table)

# Compare the original DataFrame count with the saved table count.
print("Original DataFrame records:", main_sales_df.count())
print("Saved Delta table records:", main_sales_table_df.count())

# Inspect a few records from the saved table.
main_sales_table_df.show(5, truncate=False)

# Confirm the stored column structure.
main_sales_table_df.printSchema()


Original DataFrame records: 482
Saved Delta table records: 482
+-----------+--------+----------+-------------------------------+-----------+--------+----------+----------+------------+-------------+-------------+--------------------+--------------------+
|customer_id|order_id|product_id|product_name                   |category   |quantity|unit_price|order_date|sales_amount|customer_name|city         |country             |customer_data_status|
+-----------+--------+----------+-------------------------------+-----------+--------+----------+----------+------------+-------------+-------------+--------------------+--------------------+
|CUST0031   |ORD00022|PRD013    |Memory Foam Floor Recliner     |Furniture  |1       |6300.0    |2025-06-22|6300.0      |Riya Wilson  |Visakhapatnam|India               |VALID               |
|CUST0057   |ORD00034|PRD027    |Polarized Aviator Sunglasses   |Accessories|5       |1850.0    |2025-02-10|9250.0      |Sai Davis    |Visakhapatnam|India               

In [0]:

# ------------------------------------------
# 8.3 Save the Product Exceptions Delta Table
# ------------------------------------------

# Define the destination table for product exceptions.
product_exceptions_table = (
    f"{catalog_name}.{schema_name}.product_exceptions"
)

# Save product exceptions as a managed Delta table.
product_exceptions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(product_exceptions_table)

print("Product exceptions table saved:", product_exceptions_table)


Product exceptions table saved: de_learning.sales_etl.product_exceptions


In [0]:

# ------------------------------------------
# 8.4 Validate the Product Exceptions Delta Table
# ------------------------------------------

# Read the saved product exceptions table.
product_exceptions_table_df = spark.table(
    product_exceptions_table
)

# Compare the DataFrame and saved table record counts.
print(
    "Original exception records:",
    product_exceptions_df.count()
)
print(
    "Saved Delta table records:",
    product_exceptions_table_df.count()
)

# Inspect the stored exception records.
product_exceptions_table_df.show(truncate=False)


Original exception records: 2
Saved Delta table records: 2
+-----------+--------+----------+------------+--------+--------+----------+----------+------------+-------------+-------+-------+--------------------+
|customer_id|order_id|product_id|product_name|category|quantity|unit_price|order_date|sales_amount|customer_name|city   |country|customer_data_status|
+-----------+--------+----------+------------+--------+--------+----------+----------+------------+-------------+-------+-------+--------------------+
|CUST0054   |ORD00360|PRD920    |NULL        |NULL    |3       |NULL      |2025-04-07|NULL        |Rahul Joshi  |Lucknow|India  |VALID               |
|CUST0029   |ORD00115|PRD934    |NULL        |NULL    |2       |NULL      |2025-06-04|NULL        |Sneha Nair   |Kolkata|India  |VALID               |
+-----------+--------+----------+------------+--------+--------+----------+----------+------------+-------------+-------+-------+--------------------+



In [0]:

# ------------------------------------------
# 8.5 Save the Customer Exceptions Delta Table
# ------------------------------------------

# Define the destination table for customer exceptions.
customer_exceptions_table = (
    f"{catalog_name}.{schema_name}.customer_exceptions"
)

# Save customer exceptions as a managed Delta table.
customer_exceptions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(customer_exceptions_table)

print("Customer exceptions table saved:", customer_exceptions_table)


Customer exceptions table saved: de_learning.sales_etl.customer_exceptions


In [0]:

# ------------------------------------------
# 8.6 Validate the Customer Exceptions Delta Table
# ------------------------------------------

# Read the saved customer exceptions table.
customer_exceptions_table_df = spark.table(
    customer_exceptions_table
)

# Compare the DataFrame and saved table record counts.
print(
    "Original customer exception records:",
    customer_exceptions_df.count()
)
print(
    "Saved Delta table records:",
    customer_exceptions_table_df.count()
)

# Inspect the stored customer exceptions.
customer_exceptions_table_df.show(truncate=False)


Original customer exception records: 3
Saved Delta table records: 3
+-----------+--------+----------+------------------------------+-----------+--------+----------+----------+------------------+-------------+----+-------+--------------------+
|customer_id|order_id|product_id|product_name                  |category   |quantity|unit_price|order_date|sales_amount      |customer_name|city|country|customer_data_status|
+-----------+--------+----------+------------------------------+-----------+--------+----------+----------+------------------+-------------+----+-------+--------------------+
|CUST9694   |ORD00415|PRD004    |27-inch 4K UHD Monitor        |Electronics|5       |24999.99  |2025-01-17|124999.95000000001|NULL         |NULL|NULL   |MISSING_CUSTOMER    |
|CUST9356   |ORD00210|PRD003    |Ergonomic Optical Mouse       |Electronics|4       |1299.0    |2025-06-12|5196.0            |NULL         |NULL|NULL   |MISSING_CUSTOMER    |
|CUST9679   |ORD00085|PRD010    |Adjustable Standing Desk

In [0]:

# ------------------------------------------
# 8.7 Final Delta Table Reconciliation
# ------------------------------------------

# Read all three persisted Delta tables.
main_sales_table_df = spark.table(main_sales_table)
product_exceptions_table_df = spark.table(product_exceptions_table)
customer_exceptions_table_df = spark.table(customer_exceptions_table)

# Count records in each table.
main_count = main_sales_table_df.count()
product_exception_count = product_exceptions_table_df.count()
customer_exception_count = customer_exceptions_table_df.count()

# Reconcile the non-overlapping main and product exception tables.
stored_total = main_count + product_exception_count
original_total = sales_with_customers_df.count()

print("Main sales table records:", main_count)
print("Product exceptions table records:", product_exception_count)
print("Customer exceptions table records:", customer_exception_count)
print("Total unique orders stored:", stored_total)
print("Original integrated orders:", original_total)
print("Counts match:", stored_total == original_total)


Main sales table records: 482
Product exceptions table records: 2
Customer exceptions table records: 3
Total unique orders stored: 484
Original integrated orders: 484
Counts match: True
